# 🥥 Janjang Vision - Ripeness Classification Training

Notebook ini menggabungkan 4 dataset untuk multi-class detection+classification:
1. Deteksi posisi janjang (bounding box)
2. Klasifikasi ripeness: **ripe**, **unripe**, **underripe**

Dataset yang digabungkan:
- bunchtest (237 gambar)
- nazwa (2.400+ gambar)
- gcstech (2.700+ gambar dengan ripeness labels)
- workspace-alwjv ripeness (untuk memperkaya data ripeness)

**Butuh:** Roboflow API key

In [ ]:
# 1) Install dependensi
!pip install ultralytics roboflow -q

In [ ]:
# 2) Masukkan API key Roboflow
ROBOFLOW_API_KEY = 'INPUT YOUR KEY'  # ← ganti dengan API key kamu

assert ROBOFLOW_API_KEY != 'MASUKKAN_API_KEY_KAMU_DISINI', 'Ganti API key dulu!'
print('API key OK')

In [ ]:
# 3) Download 4 dataset dari Roboflow Universe
from roboflow import Roboflow
import os, shutil

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

datasets = {}

# Dataset 1: bunchtest (237 gambar, close-up FFB)
print('=== Downloading bunchtest ===')
d1 = rf.workspace('bunchtest-33ooi').project('fresh-fruit-bunch').version(1).download('yolov8', location='ds_bunchtest')
datasets['bunchtest'] = 'ds_bunchtest'

# Dataset 2: nazwa (2.400+ gambar, scene kebun)
print('=== Downloading nazwa ===')
d2 = rf.workspace('nazwa').project('palm-oil-fruit-bunch-k').version(4).download('yolov8', location='ds_nazwa')
datasets['nazwa'] = 'ds_nazwa'

# Dataset 3: gcstech (2.700+ gambar, 5 kelas ripeness)
print('=== Downloading gcstech ===')
d3 = rf.workspace('gcstech').project('oil-palm-fruit-bunch-vlynl').version(2).download('yolov11', location='ds_gcstech')
datasets['gcstech'] = 'ds_gcstech'

# Dataset 4: ripeness classification (untuk memperkaya ripeness data)
print('=== Downloading ripeness classification ===')
try:
    d4 = rf.workspace('workspace-alwjv').project('palm-oil-ripeness-classification-iqmds').version(1).download('yolov8', location='ds_ripeness')
    datasets['ripeness'] = 'ds_ripeness'
    print('Ripeness dataset berhasil didownload!')
except Exception as e:
    print(f'Warning: Ripeness dataset download gagal ({e}). Melanjutkan dengan 3 dataset.')

print('\nSemua dataset berhasil diunduh!')
for name, path in datasets.items():
    if os.path.isdir(path):
        for sp in ['train', 'valid', 'test']:
            d = os.path.join(path, sp, 'images')
            if os.path.isdir(d):
                print(f'  {name}/{sp}: {len(os.listdir(d))} images')

In [ ]:
# 4) Gabungkan 4 dataset dengan remap kelas ripeness → 3 kategori
import os
import glob
import shutil
from pathlib import Path

# Mapping kelas ripeness ke 3 kategori
RIPENESS_MAPPING = {
    # gcstech kelas (5 kelas)
    'Decayed': None,           # skip (atau bisa set ke underripe jika mau)
    'Fully Ripe': 0,           # ripe
    'Over Ripe': 0,            # ripe
    'Partially Ripe': 1,       # unripe
    'Immature': 2,             # underripe
    'Underripe': 2,            # underripe
    
    # Default untuk dataset tanpa ripeness label
    '0': 0,  # jika hanya ada 1 class, anggap ripe
}

RIPENESS_NAMES = ['ripe', 'unripe', 'underripe']

OUT = Path('merged_ripeness')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

SPLITS = ['train', 'valid', 'test']
ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'merge_dataset',
]

DS_LIST = []
seen = set()
for root in ROOT_CANDIDATES:
    if not root.exists():
        continue
    for ds_name in ['ds_bunchtest', 'ds_nazwa', 'ds_gcstech', 'ds_ripeness']:
        ds_path = root / ds_name
        if ds_path.exists() and str(ds_path) not in seen:
            seen.add(str(ds_path))
            DS_LIST.append(ds_path)

if not DS_LIST:
    raise FileNotFoundError('Dataset tidak ditemukan')

stats = {}
for split in SPLITS:
    img_out = OUT / split / 'images'
    lbl_out = OUT / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    count = 0

    for ds in DS_LIST:
        img_dir = ds / split / 'images'
        lbl_dir = ds / split / 'labels'
        if not img_dir.is_dir():
            continue

        for img_file in sorted(img_dir.glob('*.*')):
            stem = img_file.stem
            lbl_file = lbl_dir / f'{stem}.txt'
            if not lbl_file.exists():
                continue

            try:
                with open(lbl_file, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
            except Exception:
                continue

            if not lines:
                continue

            new_lines = []
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    old_class = int(parts[0])
                    
                    # Cek apakah dataset punya ripeness info (dari gcstech)
                    ripeness_class = None
                    if ds.name == 'ds_gcstech':
                        # gcstech punya 5 kelas ripeness
                        ripeness_map_gcstech = {
                            0: 2,  # Decayed -> skip (None)
                            1: 0,  # Fully Ripe -> ripe
                            2: 2,  # Immature -> underripe
                            3: 0,  # Over Ripe -> ripe
                            4: 1,  # Partially Ripe -> unripe
                        }
                        ripeness_class = ripeness_map_gcstech.get(old_class, 0)
                    elif ds.name == 'ds_ripeness':
                        # ripeness dataset sudah punya class ripeness
                        ripeness_class = old_class if old_class < 3 else 0
                    else:
                        # bunchtest & nazwa tidak punya ripeness, default ripe
                        ripeness_class = 0
                    
                    if ripeness_class is not None:
                        parts[0] = str(ripeness_class)
                        new_lines.append(' '.join(parts))

            if not new_lines:
                continue

            dest_img = img_out / f'{ds.name}_{img_file.name}'
            dest_lbl = lbl_out / f'{ds.name}_{stem}.txt'
            shutil.copy2(img_file, dest_img)
            with open(dest_lbl, 'w', encoding='utf-8') as f:
                f.write('\n'.join(new_lines))
            count += 1

    stats[split] = count

base_path = os.path.abspath(str(OUT))
yaml_content = f'''path: {base_path}
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['ripe', 'unripe', 'underripe']
'''
(OUT / 'data.yaml').write_text(yaml_content, encoding='utf-8')

print('Dataset gabungan dengan ripeness classification siap!')
print(f'  train: {stats["train"]} gambar')
print(f'  valid: {stats["valid"]} gambar')
print(f'  test:  {stats["test"]} gambar')
print('\nKelas (3):  ripe (0), unripe (1), underripe (2)')
print('\ndata.yaml:')
print((OUT / 'data.yaml').read_text(encoding='utf-8'))

In [ ]:
# 5) Training model dengan ripeness classification!
!python janjang_counter.py --train merged_ripeness/data.yaml --epochs 100 --imgsz 640 --batch 16 --patience 25

In [ ]:
# 6) Download model hasil
import os
import glob
import shutil
from pathlib import Path

models = sorted(glob.glob('runs/detect/*/weights/best.pt'))
if models:
    best_model = models[-1]
    print('Model ditemukan:', best_model)

    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle').exists():
        from IPython.display import FileLink, display
        print('Environment: Kaggle')
        print('Gunakan link di bawah untuk download model:')
        display(FileLink(best_model))
    else:
        try:
            from google.colab import files
            files.download(best_model)
            print('File berhasil didownload via Colab.')
        except Exception:
            target = os.path.join(os.getcwd(), os.path.basename(best_model))
            shutil.copy2(best_model, target)
            print(f'File disalin ke: {target}')
else:
    print('Model belum ada - training mungkin belum selesai')

## Setelah dapat model ripeness

Model yang dihasilkan sudah dilengkapi dengan 3 kelas ripeness:
- `ripe` (0): Fully ripe / ready to harvest
- `unripe` (1): Partially ripe / still developing
- `underripe` (2): Immature / not ready yet

Gunakan model dengan:

```bash
python janjang_counter.py foto.jpg --model best_ripeness.pt
```

Output akan menampilkan:
- Bounding box untuk setiap janjang
- Label ripeness (ripe/unripe/underripe) di atas box
- Count per kelas ripeness
```